
# Experiment 2 — Segment Length Scan

 - Subject: 2

 - Channel: Fz

 - Loss: MSE

 - Segment lengths: 0.5, 1, 2, 3 s

 - CV: GroupKFold, 5 folds over 40 trials

 - Primary inference unit: 40 held-out trials

 - Frequency range: 4–25 Hz

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, r'D:\Thesis\Codes\Source')

from face2eeg.stats import (
    summarize_trials,
    analyze_segment_lengths_trial_level,
    segment_lengths_to_dataframes,
)

from face2eeg.plotting import (
    plot_band_comparison,
    plot_mean_curve_with_ci,
    plot_segment_metric_curves,
)

from face2eeg.analysis import (
    setup,
    experiment_paths,
    load_result,
    validate_trial_alignment,
)

from face2eeg.core.config import SEED

%matplotlib inline

In [ ]:
# %%
BASE = r'D:\Thesis\Codes\Data\Experiments\Cross-Validation'

base = setup(BASE)
exp = experiment_paths(base, 'Segment-Lengths')

RES_DIR = exp['results']
FIG_DIR = exp['figures']

os.makedirs(FIG_DIR, exist_ok=True)

TABLE_DIR = os.path.join(
    RES_DIR,
    'Analysis-Tables'
)
os.makedirs(TABLE_DIR, exist_ok=True)

N_BOOT = 10_000
N_PERM = 10_000

print("RES_DIR   :", RES_DIR)
print("FIG_DIR   :", FIG_DIR)
print("TABLE_DIR :", TABLE_DIR)

## 1. Load results


In [ ]:
variants = {
    0.5: 'tw0.5',
    1.0: 'tw1',
    2.0: 'tw2',
    3.0: 'tw3',
}

r_by_T = {
    T: load_result(
        RES_DIR,
        subject=2,
        channel='Fz',
        variant=variant,
    )
    for T, variant in variants.items()
}

for T, result in r_by_T.items():
    print(f"\nT = {T}s")
    print(result.summary())

## 2. Extract trial-level arrays + alignment

In [ ]:
y_true_by_T = {}
y_pred_by_T = {}
trial_ids_by_T = {}

for T, result in r_by_T.items():
    yt, yp = result.get_all_trials()

    y_true_by_T[T] = yt
    y_pred_by_T[T] = yp
    trial_ids_by_T[T] = result.get_trial_indices()

    print(
        f"T={T}: "
        f"y_true={yt.shape}, "
        f"y_pred={yp.shape}, "
        f"trials={len(trial_ids_by_T[T])}"
    )

In [ ]:
T_ref = 0.5
ref_ids = trial_ids_by_T[T_ref]
ref_true = y_true_by_T[T_ref]

for T in variants:
    validate_trial_alignment(
        ref_ids,
        trial_ids_by_T[T],
        name_a=f"T={T_ref}",
        name_b=f"T={T}",
    )

    if not np.allclose(
        ref_true,
        y_true_by_T[T],
        equal_nan=True,
    ):
        raise ValueError(
            f"EEG targets differ between T={T_ref} and T={T}."
        )

print("Trial IDs and EEG targets are aligned across all segment lengths.")

## 3. Trial-level summaries

In [ ]:
summaries_by_T = {
    T: summarize_trials(
        y_true_by_T[T],
        y_pred_by_T[T],
        seed=SEED,
        n_boot=N_BOOT
    )
    for T in variants
}

for T in variants:
    s = summaries_by_T[T]
    print(
        f"T={T:.1f}s | "
        f"r={s['r_mean']:+.4f} ± {s['r_std']:.4f} | "
        f"RMSE={s['rmse_mean']:.4f}"
    )

## 4. Descriptive summary table

In [ ]:
rows = []

for T, s in summaries_by_T.items():
    rows.append({
        "Segment Length (s)": T,
        "Fisher-z Aggregated Pearson r": s["r_pooled"],
        "95% Bootstrap CI (r)": s["r_pooled_ci_boot"],
        "Mean Trial-level RMSE": s["rmse_mean"],
        "95% Bootstrap CI (RMSE)": s["rmse_trial_ci_boot"],
    })

df_summary = pd.DataFrame(rows).sort_values("Segment Length (s)")

display(df_summary.round(2))

In [ ]:
df_paper = pd.DataFrame({
    "Segment Length (s)": df_summary["Segment Length (s)"],
    "Fisher-z Aggregated Pearson r":
        df_summary["Fisher-z Aggregated Pearson r"].map(lambda x: f"{x:.2f}"),
    "95% Bootstrap CI (r)": [
        f"[{lo:.2f}, {hi:.2f}]"
        for lo, hi in df_summary["95% Bootstrap CI (r)"]
    ],
    "Mean Trial-level RMSE":
        df_summary["Mean Trial-level RMSE"].map(lambda x: f"{x:.2f}"),
    "95% Bootstrap CI (RMSE)": [
        f"[{lo:.2f}, {hi:.2f}]"
        for lo, hi in df_summary["95% Bootstrap CI (RMSE)"]
    ],
})

display(df_paper)

In [ ]:
df_summary.to_csv(
    os.path.join(TABLE_DIR, "exp2_descriptive_summary.csv"),
    index=False,
)

df_summary.to_latex(
    os.path.join(TABLE_DIR, "exp2_descriptive_summary.tex"),
    index=False,
    float_format="%.3f",
)

In [ ]:
variants

In [ ]:
plot_band_comparison(
    {
        "a": summaries_by_T[0.5],
        "b": summaries_by_T[1],
        "c": summaries_by_T[2],
        "d": summaries_by_T[3],
    },
    {
        "a": "T=0.5",
        "b": "T=1",
        "c": "T=2",
        "d": "T=3",
    },
    layout="side",
    error="ci_boot",
    save_fig=True,
    file_name=os.path.join(FIG_DIR, 'exp2_band_comparison.pdf'),
)


In [ ]:
def extract_per_condition_curves(
    summaries_dict,
    bands=("theta", "alpha", "beta"),
):
    """
    Build per-condition mean + CI tables for plotting from a dict of
    `summarize_trials` outputs — one summary per condition.

    Pure reshaping. No statistics are recomputed here: RMSE, Fisher-z
    pooled r, and per-band coherence with their bootstrap CIs are all
    read directly from the summary dicts.

    Parameters
    ----------
    summaries_dict : dict of {condition_label: summary}
        Each value is the output of `summarize_trials`.
    bands : tuple of str
        Coherence bands to extract.

    Returns
    -------
    dict with keys:
        "rmse" : DataFrame
        "r"    : DataFrame
        "coh"  : dict of {band: DataFrame}

    Each DataFrame has columns:
        condition_idx, condition, mean, ci_low, ci_high
    """
    conditions = list(summaries_dict.keys())

    rows_rmse = []
    rows_r    = []
    rows_coh  = {b: [] for b in bands}

    for j, cond in enumerate(conditions):
        s = summaries_dict[cond]

        # ---- RMSE (read directly) ----
        rows_rmse.append(dict(
            condition_idx=j,
            condition=cond,
            mean=s["rmse_mean"],
            ci_low=s["rmse_trial_ci_boot"][0],
            ci_high=s["rmse_trial_ci_boot"][1],
        ))

        # ---- Pearson r (Fisher-z pooled, already back-transformed) ----
        rows_r.append(dict(
            condition_idx=j,
            condition=cond,
            mean=s["r_pooled"],
            ci_low=s["r_pooled_ci_boot"][0],
            ci_high=s["r_pooled_ci_boot"][1],
        ))

        # ---- Coherence per band (read directly) ----
        for b in bands:
            ms = s["band_summary"][b]["coh"]["mean_sd_ci"]
            cb = s["band_summary"][b]["coh"]["ci_boot"]
            rows_coh[b].append(dict(
                condition_idx=j,
                condition=cond,
                mean=ms["mean"],
                ci_low=cb[0],
                ci_high=cb[1],
            ))

    return {
        "rmse": pd.DataFrame(rows_rmse),
        "r":    pd.DataFrame(rows_r),
        "coh":  {b: pd.DataFrame(rows_coh[b]) for b in bands},
    }

In [ ]:

T_labels = {
    "T=0.5": summaries_by_T[0.5],
    "T=1":   summaries_by_T[1.0],
    "T=2":   summaries_by_T[2.0],
    "T=3":   summaries_by_T[3.0],
}

results = analyze_segment_lengths_trial_level(
    summaries_dict=T_labels,
    seed=SEED,
    n_perm=N_PERM,
    n_boot=N_BOOT,
)

curves = extract_per_condition_curves(T_labels, bands=("theta", "alpha", "beta"))

In [ ]:
for metric in ("rmse", "r_z"):
    M = results[metric]["matrix"]
    print(
        f"{metric}: shape={M.shape}, "
        f"complete trials={np.sum(np.all(np.isfinite(M), axis=1))}"
    )

for band in ("theta", "alpha", "beta"):
    for metric in ("coherence", "rbp_error"):
        M = results[metric]["per_band"][band]["matrix"]
        print(
            f"{metric} / {band}: "
            f"shape={M.shape}, "
            f"complete trials={np.sum(np.all(np.isfinite(M), axis=1))}"
        )

In [ ]:
df_omni, df_pairwise = segment_lengths_to_dataframes(results)

display(df_omni)
display(df_pairwise.round(4))

In [ ]:
df_omni.to_csv(
    os.path.join(TABLE_DIR, "exp2_omnibus.csv"),
    index=False,
)

df_pairwise.to_csv(
    os.path.join(TABLE_DIR, "exp2_pairwise.csv"),
    index=False,
)

df_omni.to_latex(
    os.path.join(TABLE_DIR, "exp2_omnibus.tex"),
    index=False,
    float_format="%.4f",
)

df_pairwise.to_latex(
    os.path.join(TABLE_DIR, "exp2_pairwise.tex"),
    index=False,
    float_format="%.4f",
)

In [ ]:
plot_mean_curve_with_ci(
    {f"T={T:g}s": summaries_by_T[T] for T in variants},
    what="coh",
    freq_range=(4, 25),
    save_fig=True,
    file_name=os.path.join(
        FIG_DIR,
        "exp2_coh_mean_curve_with_ci.pdf",
    ),
)

In [ ]:
plot_mean_curve_with_ci(
    {f"T={T:g}s": summaries_by_T[T] for T in variants},
    what="psd_pred",
    freq_range=(4, 25),
    show_true_if_pred=True,
    save_fig=True,
    file_name=os.path.join(
        FIG_DIR,
        "exp2_psd_pred_mean_curve_with_ci.pdf",
    ),
)

In [ ]:
figs_main = plot_segment_metric_curves(
    df_rmse=curves["rmse"],
    df_r=curves["r"],
    labels=("T=0.5 s", "T=1 s", "T=2 s", "T=3 s"),
    save_fig=True,
    file_name=os.path.join(FIG_DIR, "exp2_RMSE_and_R.pdf"),
)
plt.show()

In [ ]:
figs_coh = plot_segment_metric_curves(
    df_per_band=curves["coh"],
    labels=("T=0.5 s", "T=1 s", "T=2 s", "T=3 s"),
    ylabel="Coherence",
    ylim=(0, 1),
    save_fig=True,
    file_name=os.path.join(FIG_DIR, "exp2_Coherence_per_band.pdf"),
)
plt.show()